In [59]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../../amazon_electronics.db")
df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

df.head()


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating_count,about_product,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1.099,0.64,24269,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.000,0.43,43994,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1.899,0.90,7928,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.000,0.53,94363,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.000,0.61,16905,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [60]:
df["category"] = df["category"].str.split(r"[|&]")
df["category"] = df["category"].apply(lambda x: list(set(x)))

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
category_matrix = mlb.fit_transform(df["category"])

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df[["discounted_price"]])

In [61]:
feature_matrix = np.hstack((category_matrix, price_scaled))

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

In [62]:
product_index_map = pd.Series(
    df.index,
    index=df["product_id"]
).to_dict()


def recommend_products(product_id, similarity_matrix, df, product_index_map, top_n=5):

    if product_id not in product_index_map:
        return "Product ID not found"

    index = product_index_map[product_id]

    similarity_scores = list(enumerate(similarity_matrix[index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_products = similarity_scores[1:top_n+1]

    recommended_indices = [i[0] for i in top_products]

    return df.iloc[recommended_indices][
        ["product_id", "product_name", "discounted_price"]
    ]

In [64]:
recommend_products(
    "B09X79PP8F",
    similarity_matrix,
    df,
    product_index_map
)

,product_id,product_name,discounted_price
51,B0711PVX6Z,AmazonBasics Micro USB Fast Charging Cable for...,179.00
76,B09YLXYP7Y,Ambrane 60W / 3A Fast Charging Output Cable wi...,179.00
133,B09X79PP8F,MI 2-in-1 USB Type C Cable (Micro USB to Type ...,179.00
240,B09YLX91QR,Ambrane 60W / 3A Fast Charging Output Cable wi...,179.00
6,B08WRWPM22,"boAt Micro USB 55 Tangle-free, Sturdy Micro US...",176.63
